# LeetCode 1271 — Hexspeak bằng Python

**Link đề:** https://leetcode.com/problems/hexspeak/
**Độ khó:** Easy · **Chủ đề:** Math, String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def toHexspeak(self, num: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **hệ thập lục phân (hexadecimal, cơ số 16)** và cách đổi một số thập phân sang hex **bằng tay**.
2. Dùng công cụ có sẵn: `int(...)`, `hex(...)`, `format(n, "X")`, `str.upper`, `str.replace`, `str.translate`.
3. Tự cài đặt phép đổi cơ số bằng `divmod(n, 16)` — kỹ thuật "lấy chữ số cuối" tổng quát hóa từ `% 10`.
4. Kiểm tra "chuỗi chỉ gồm ký tự hợp lệ" bằng `set` và `all(...)`, có **dừng sớm** khi gặp chữ số xấu.
5. Trace, chứng minh, test đối chiếu, lỗi hay gặp, biến thể và bài tập có đáp án.

## 1. Đề bài và ví dụ

Một số thập phân được đổi sang dạng **Hexspeak** như sau:
1. Đổi số sang chuỗi **hexadecimal viết HOA**.
2. Thay mọi chữ số `'0'` bằng chữ `'O'`, mọi chữ số `'1'` bằng chữ `'I'`.

Dạng Hexspeak là **hợp lệ** khi và chỉ khi nó chỉ gồm các chữ cái trong tập `{'A', 'B', 'C', 'D', 'E', 'F', 'I', 'O'}`.

Cho chuỗi `num` biểu diễn số nguyên thập phân `N`, trả về Hexspeak của `N` nếu hợp lệ, ngược lại trả về `"ERROR"`.

| `num` | hex | Output |
|---|---|---|
| `"257"` | `101` | `"IOI"` |
| `"3"` | `3` | `"ERROR"` |

**Constraints (theo hiểu biết về đề):**
- `1 <= num.length <= 12`
- `num` không có số 0 ở đầu.
- `num` biểu diễn số nguyên trong đoạn `[1, 10^12]`.

Chú ý: **input là chuỗi** (`str`), không phải `int`.

### Vài ví dụ "vui" để có cảm giác

Lập trình viên hay dùng hex để viết "chữ": `0xDEADBEEF`, `0xC0FFEE`, `0xB0BA`… Thay `0 → O`, `1 → I` thì đọc được thành từ:

| thập phân | hex | Hexspeak |
|---:|---|---|
| 45242 | `B0BA` | `BOBA` |
| 12648430 | `C0FFEE` | `COFFEE` |
| 3735928559 | `DEADBEEF` | `DEADBEEF` |
| 497 | `1F1` | `IFI` |
| 16 | `10` | `IO` |
| 26 | `1A` | `IA` |
| 18 | `12` | `ERROR` (có chữ số `2`) |

In [1]:
for v in [45242, 12648430, 3735928559, 497, 16, 26, 18]:
    print(v, "->", format(v, "X"))

45242 -> B0BA
12648430 -> C0FFEE
3735928559 -> DEADBEEF
497 -> 1F1
16 -> 10
26 -> 1A
18 -> 12


## 2. Nền tảng: hệ thập lục phân (cơ số 16)

### 2.1. Cơ số 10 nhắc lại

`257 = 2·10² + 5·10¹ + 7·10⁰`. Mỗi vị trí có "trọng số" là lũy thừa của 10, mỗi chữ số từ `0` đến `9`.

### 2.2. Cơ số 16

Mỗi vị trí có trọng số là lũy thừa của **16**, mỗi chữ số từ `0` đến `15`. Vì chỉ có 10 ký hiệu số, ta mượn chữ cái:

| giá trị | 0–9 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|
| ký hiệu | `0`–`9` | `A` | `B` | `C` | `D` | `E` | `F` |

Ví dụ: `0x101 = 1·16² + 0·16¹ + 1·16⁰ = 256 + 0 + 1 = 257`.

In [2]:
print(1 * 16**2 + 0 * 16 + 1)      # 257
print(int("101", 16))              # 257: đọc chuỗi "101" theo cơ số 16
print(int("FF", 16), int("ff", 16))   # 255 255
print(0x101)                       # 257: Python cho viết hằng hex bằng tiền tố 0x

257
257
255 255
257


### 2.3. Đổi chuỗi → số: `int(num)`

Input là `str`, phải đổi sang `int` trước khi làm toán. `int("257")` → `257`. Có thể truyền cơ số thứ hai: `int("101", 16)`.

In [3]:
num = "257"
print(type(num), type(int(num)))
print(int(num) + 1)                # 258
print(num + "1")                   # "2571" — cộng chuỗi là NỐI chuỗi!

<class 'str'> <class 'int'>
258
2571


### 2.4. Số → chuỗi hex: `hex()` và `format(n, "X")`

- `hex(257)` → `'0x101'`: có tiền tố `0x`, chữ **thường**.
- `format(257, "X")` → `'101'`: không tiền tố, chữ **HOA**. (`"x"` thường cho chữ thường.)
- Hoặc f-string: `f"{257:X}"`.

In [4]:
n = 700896
print(hex(n))                 # '0xab1e0'
print(hex(n)[2:])             # 'ab1e0'  (cắt bỏ 2 ký tự '0x')
print(hex(n)[2:].upper())     # 'AB1E0'
print(format(n, "X"))         # 'AB1E0'
print(f"{n:X}", f"{n:x}")     # AB1E0 ab1e0

0xab1e0
ab1e0
AB1E0
AB1E0
AB1E0 ab1e0


### 2.5. Thay ký tự: `replace` và `translate`

- `s.replace("0", "O")` trả **chuỗi mới** (chuỗi Python bất biến — immutable), thay mọi `"0"`.
- Gọi nối tiếp: `s.replace("0", "O").replace("1", "I")`.
- `str.maketrans("01", "OI")` tạo bảng dịch `0→O`, `1→I`; `s.translate(bảng)` thay **cùng lúc** nhiều ký tự trong 1 lượt.

In [5]:
s = "AB1E0"
print(s.replace("0", "O"))                       # AB1EO
print(s.replace("0", "O").replace("1", "I"))     # ABIEO
table = str.maketrans("01", "OI")
print(s.translate(table))                        # ABIEO
print(s)                                         # AB1E0 — chuỗi gốc không đổi

AB1EO
ABIEO
ABIEO
AB1E0


### 2.6. Kiểm tra "chỉ gồm ký tự hợp lệ": `set` và `all`

- `set("IOI")` → `{'I', 'O'}`: tập các ký tự **khác nhau**.
- `A <= B` với hai `set` nghĩa là "A là **tập con** của B".
- `all(điều_kiện for c in s)` trả `True` nếu **mọi** ký tự thỏa (và dừng ngay khi gặp ký tự sai).

In [6]:
VALID = set("ABCDEFIO")
print(set("IOI"), set("IOI") <= VALID)       # {'I','O'} True (thứ tự in có thể khác)
print(set("I3") <= VALID)                    # False
print(all(c in VALID for c in "ABIEO"))      # True
print(all(c in VALID for c in "AB2EO"))      # False

{'I', 'O'} True
False
True
False


## 3. Cách 1 — Dùng hàm có sẵn (đúng theo từng bước của đề)

1. `n = int(num)`
2. `h = format(n, "X")` (hoặc `hex(n)[2:].upper()`)
3. `h = h.replace("0", "O").replace("1", "I")`
4. Nếu mọi ký tự thuộc `VALID` → trả `h`, ngược lại `"ERROR"`.

In [7]:
VALID = set("ABCDEFIO")

def hexspeak_builtin(num: str) -> str:
    h = hex(int(num))[2:].upper()
    h = h.replace("0", "O").replace("1", "I")
    return h if set(h) <= VALID else "ERROR"

print(hexspeak_builtin("257"))        # IOI
print(hexspeak_builtin("3"))          # ERROR
print(hexspeak_builtin("12648430"))   # COFFEE

IOI
ERROR
COFFEE


`A if điều_kiện else B` là **biểu thức điều kiện** (toán tử ba ngôi): nếu đúng trả `A`, sai trả `B`.

**Độ phức tạp:** gọi `d` = số chữ số hex của `N` (`d ≈ log₁₆ N`, tối đa 10 với `N ≤ 10^12`). Mọi bước đều `O(d)` thời gian, `O(d)` bộ nhớ (cộng thêm chi phí `int(num)` với độ dài chuỗi `L ≤ 12`).

## 4. Nút thắt — hiểu cái "hộp đen" `hex()`

Cách 1 rất ngắn, nhưng:
- Phỏng vấn có thể yêu cầu **tự đổi cơ số**.
- Ta tạo cả chuỗi rồi mới phát hiện lỗi; nếu tự đổi, có thể **dừng ngay** khi gặp chữ số `2..9`.

Vậy làm sao lấy từng chữ số hex của một số?

## 5. Insight — `% 16` và `// 16` giống `% 10` và `// 10`

Với cơ số 10: `n % 10` là chữ số cuối, `n // 10` bỏ chữ số cuối. Cơ số 16 y hệt:

- `n % 16` = chữ số hex **cuối** (0..15)
- `n // 16` = bỏ chữ số hex cuối

`divmod(n, 16)` trả về cả hai: `(n // 16, n % 16)`. Lặp đến khi `n == 0`, ta nhận các chữ số **từ phải sang trái** → cuối cùng đảo ngược lại.

```text
257: divmod(257, 16) = (16, 1)   -> chữ số 1
 16: divmod(16, 16)  = (1, 0)    -> chữ số 0
  1: divmod(1, 16)   = (0, 1)    -> chữ số 1
đọc ngược: 1 0 1  -> "101"
```

In [8]:
n = 257
digits = []
while n > 0:
    n, d = divmod(n, 16)
    digits.append(d)
    print("n =", n, "| chữ số vừa lấy =", d)
print("từ phải sang trái:", digits, "-> đúng thứ tự:", digits[::-1])

n = 16 | chữ số vừa lấy = 1
n = 1 | chữ số vừa lấy = 0
n = 0 | chữ số vừa lấy = 1
từ phải sang trái: [1, 0, 1] -> đúng thứ tự: [1, 0, 1]


### Ánh xạ giá trị chữ số → ký tự Hexspeak

Chỉ các giá trị sau là hợp lệ:

| giá trị | 0 | 1 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|---|
| ký tự | `O` | `I` | `A` | `B` | `C` | `D` | `E` | `F` |

Các giá trị `2..9` → **ERROR** ngay. Ta có thể lưu bằng một `dict`, hoặc dùng chuỗi tra cứu `"0123456789ABCDEF"[d]`.

In [9]:
TO_CHAR = {0: "O", 1: "I", 10: "A", 11: "B", 12: "C", 13: "D", 14: "E", 15: "F"}
print(TO_CHAR[0], TO_CHAR[10], TO_CHAR.get(5))   # O A None  (.get trả None nếu không có key)
print(5 in TO_CHAR, 15 in TO_CHAR)               # False True
HEX = "0123456789ABCDEF"
print(HEX[10], HEX[15], HEX[1])                  # A F 1

O A None
False True
A F 1


## 6. Cách 2 — Tự đổi cơ số + dừng sớm (lời giải chính)

In [10]:
TO_CHAR = {0: "O", 1: "I", 10: "A", 11: "B", 12: "C", 13: "D", 14: "E", 15: "F"}

def hexspeak_manual(num: str) -> str:
    n = int(num)
    chars = []
    while n > 0:
        n, d = divmod(n, 16)
        if d not in TO_CHAR:          # chữ số 2..9 -> không hợp lệ, dừng ngay
            return "ERROR"
        chars.append(TO_CHAR[d])
    return "".join(reversed(chars))

print(hexspeak_manual("257"))         # IOI
print(hexspeak_manual("3"))           # ERROR
print(hexspeak_manual("45242"))       # BOBA
print(hexspeak_manual("3735928559"))  # DEADBEEF

IOI
ERROR
BOBA
DEADBEEF


### Giải thích `"".join(reversed(chars))`

- `reversed(chars)` cho các phần tử theo thứ tự ngược (không tạo list mới ngay).
- `"".join(...)` nối các chuỗi con lại, xen giữa bằng chuỗi rỗng `""`.
- Vì sao không cộng chuỗi `res = c + res` trong vòng lặp? Được, nhưng mỗi lần cộng tạo chuỗi mới → với chuỗi dài sẽ tốn `O(d²)`. Thói quen tốt: gom vào list rồi `join`.

In [11]:
chars = ["I", "O", "I", "A"]
print(list(reversed(chars)))        # ['A', 'I', 'O', 'I']
print("".join(reversed(chars)))     # AIOI
print("-".join(["x", "y", "z"]))    # x-y-z

['A', 'I', 'O', 'I']
AIOI
x-y-z


## 7. Cách 3 — `translate` + kiểm tra bằng `str.isalpha`/tập hợp

Biến thể gọn của Cách 1: sau khi thay `0→O`, `1→I`, mọi ký tự còn là **chữ số** (`2..9`) đều xấu. Vì `A..F, I, O` đều là chữ cái, chỉ cần kiểm tra chuỗi **toàn chữ cái** bằng `str.isalpha()`.

In [12]:
TABLE = str.maketrans("01", "OI")

def hexspeak_translate(num: str) -> str:
    h = format(int(num), "X").translate(TABLE)
    return h if h.isalpha() else "ERROR"

print("I3".isalpha(), "IOI".isalpha())   # False True
print(hexspeak_translate("257"), hexspeak_translate("3"), hexspeak_translate("700896"))

False True
IOI ERROR ABIEO


Vì sao `isalpha()` đủ? Chuỗi hex in HOA chỉ chứa `0-9` và `A-F`. Sau khi đổi `0, 1` thành chữ, ký tự không phải chữ cái chỉ có thể là `2..9` — đúng các ký tự gây ERROR.

## 8. Cách 4 — Kiểm tra trước, đổi sau (không cần `replace`)

Quan sát: Hexspeak hợp lệ ⇔ hex của `N` **không chứa** chữ số `2..9`. Ta có thể kiểm tra bằng `any(...)` trên chuỗi hex, rồi mới dịch.

In [13]:
def hexspeak_check_first(num: str) -> str:
    h = format(int(num), "X")
    if any(c in "23456789" for c in h):
        return "ERROR"
    return h.replace("0", "O").replace("1", "I")

print(any([False, True]), any([False, False]))   # True False
print(hexspeak_check_first("257"), hexspeak_check_first("18"), hexspeak_check_first("26"))

True False
IOI ERROR IA


## 9. Trace từng bước (Cách 2)

`num = "700896"` (hex `AB1E0`):

| vòng | `n` trước | `divmod(n, 16)` | `d` | hợp lệ? | `chars` sau |
|---:|---:|---|---:|---|---|
| 1 | 700896 | (43806, 0) | 0 | ✔ `O` | `[O]` |
| 2 | 43806 | (2737, 14) | 14 | ✔ `E` | `[O, E]` |
| 3 | 2737 | (171, 1) | 1 | ✔ `I` | `[O, E, I]` |
| 4 | 171 | (10, 11) | 11 | ✔ `B` | `[O, E, I, B]` |
| 5 | 10 | (0, 10) | 10 | ✔ `A` | `[O, E, I, B, A]` |

Đảo ngược → `"ABIEO"`. Ví dụ `num = "18"`: vòng 1 `divmod(18,16) = (1, 2)` → `d = 2` → **ERROR** ngay.

In [14]:
def trace_hexspeak(num: str) -> str:
    n = int(num)
    chars = []
    step = 1
    while n > 0:
        before = n
        n, d = divmod(n, 16)
        ok = d in TO_CHAR
        print(f"vòng {step}: n={before:>7} -> divmod={(n, d)} | d={d:>2} | {'OK ' + TO_CHAR[d] if ok else 'XẤU -> ERROR'}")
        if not ok:
            return "ERROR"
        chars.append(TO_CHAR[d])
        print(f"          chars={chars}")
        step += 1
    res = "".join(reversed(chars))
    print("kết quả:", res)
    return res

trace_hexspeak("700896")
print()
trace_hexspeak("18")

vòng 1: n= 700896 -> divmod=(43806, 0) | d= 0 | OK O
          chars=['O']
vòng 2: n=  43806 -> divmod=(2737, 14) | d=14 | OK E
          chars=['O', 'E']
vòng 3: n=   2737 -> divmod=(171, 1) | d= 1 | OK I
          chars=['O', 'E', 'I']
vòng 4: n=    171 -> divmod=(10, 11) | d=11 | OK B
          chars=['O', 'E', 'I', 'B']
vòng 5: n=     10 -> divmod=(0, 10) | d=10 | OK A
          chars=['O', 'E', 'I', 'B', 'A']
kết quả: ABIEO

vòng 1: n=     18 -> divmod=(1, 2) | d= 2 | XẤU -> ERROR


'ERROR'

## 10. Code nộp LeetCode

In [15]:
class Solution:
    def toHexspeak(self, num: str) -> str:
        to_char = {0: "O", 1: "I", 10: "A", 11: "B", 12: "C", 13: "D", 14: "E", 15: "F"}
        n = int(num)
        chars = []
        while n > 0:
            n, d = divmod(n, 16)
            if d not in to_char:
                return "ERROR"
            chars.append(to_char[d])
        return "".join(reversed(chars))

print(Solution().toHexspeak("257"))   # IOI
print(Solution().toHexspeak("3"))     # ERROR

IOI
ERROR


Bản ngắn dùng thư viện (cũng hợp lệ):

```python
class Solution:
    def toHexspeak(self, num: str) -> str:
        h = format(int(num), "X").replace("0", "O").replace("1", "I")
        return h if set(h) <= set("ABCDEFIO") else "ERROR"
```

## 11. Vì sao đúng?

- **Đổi cơ số đúng:** mỗi vòng giữ bất biến `N_gốc = n · 16^k + (giá trị các chữ số đã lấy)`, với `k` là số chữ số đã lấy. Khi `n = 0`, các chữ số đã lấy chính là biểu diễn hex của `N` (theo thứ tự ngược). Vì `N >= 1`, vòng lặp chạy ít nhất 1 lần và không sinh số 0 thừa ở đầu.
- **Ánh xạ đúng đề:** `0→O`, `1→I`, `10..15 → A..F` đúng là "viết HOA rồi thay 0, 1". Mọi giá trị khác (`2..9`) sẽ sinh ký tự số không thuộc tập hợp lệ → ERROR.
- **Dừng sớm an toàn:** chỉ cần một chữ số xấu là kết quả chắc chắn ERROR, nên trả về ngay không làm sai kết quả.

## 12. Độ phức tạp và so sánh

Gọi `d = số chữ số hex ≈ log₁₆(N)` (với `N ≤ 10^12` thì `d ≤ 10`).

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. `hex` + `replace` + `set` | `O(d)` | `O(d)` | ngắn, đúng từng bước đề |
| 2. `divmod` thủ công | `O(d)` | `O(d)` | tự cài đặt, dừng sớm |
| 3. `translate` + `isalpha` | `O(d)` | `O(d)` | 1 lượt thay ký tự |
| 4. Kiểm tra trước (`any`) | `O(d)` | `O(d)` | tách "kiểm tra" và "dịch" |

(Chưa tính `int(num)` — đổi chuỗi `L` ký tự sang số, `O(L)` với `L ≤ 12`.)

## 13. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.toHexspeak("257"), "IOI")
check("đề 2", sol.toHexspeak("3"), "ERROR")
check("nhỏ nhất 1", sol.toHexspeak("1"), "I")
check("10 -> A", sol.toHexspeak("10"), "A")
check("15 -> F", sol.toHexspeak("15"), "F")
check("16 -> IO", sol.toHexspeak("16"), "IO")
check("có số 0 bên trong", sol.toHexspeak("45242"), "BOBA")
check("DEADBEEF", sol.toHexspeak("3735928559"), "DEADBEEF")
check("chữ số xấu ở giữa", sol.toHexspeak("291"), "ERROR")       # 0x123
check("10^12", sol.toHexspeak(str(10**12)), "ERROR")             # E8D4A51000 có số 8

PASS đề 1: IOI
PASS đề 2: ERROR
PASS nhỏ nhất 1: I
PASS 10 -> A: A
PASS 15 -> F: F
PASS 16 -> IO: IO
PASS có số 0 bên trong: BOBA
PASS DEADBEEF: DEADBEEF
PASS chữ số xấu ở giữa: ERROR
PASS 10^12: ERROR


In [17]:
import random
random.seed(1271)
methods = [hexspeak_builtin, hexspeak_manual, hexspeak_translate, hexspeak_check_first]
samples = [str(random.randint(1, 10**12)) for _ in range(500)]
# Thêm các số chắc chắn hợp lệ: ghép ngẫu nhiên từ các chữ số hex 0,1,A..F
good_digits = "01ABCDEF"
for _ in range(300):
    h = random.choice("1ABCDEF") + "".join(random.choice(good_digits) for _ in range(random.randint(0, 8)))
    samples.append(str(int(h, 16)))
cnt_ok = 0
for s in samples:
    expected = hexspeak_builtin(s)
    for f in methods:
        assert f(s) == expected, (f.__name__, s)
    assert Solution().toHexspeak(s) == expected
    cnt_ok += expected != "ERROR"
print(f"{len(samples)} mẫu: 4 cách + Solution khớp; trong đó {cnt_ok} mẫu hợp lệ")

800 mẫu: 4 cách + Solution khớp; trong đó 300 mẫu hợp lệ


## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên cắt `'0x'` khi dùng `hex()`

`hex(257)` là `'0x101'`. Nếu không cắt, chữ `x` (thường) và số `0` đầu làm kết quả sai.

In [18]:
def wrong_keep_prefix(num):
    h = hex(int(num)).upper().replace("0", "O").replace("1", "I")
    return h if set(h) <= set("ABCDEFIO") else "ERROR"

print("sai :", wrong_keep_prefix("257"))    # 'OXIOI' có X -> ERROR
print("đúng:", Solution().toHexspeak("257"))

sai : ERROR
đúng: IOI


### Lỗi 2: Quên `.upper()`

`hex()` cho chữ **thường** (`'ab1e0'`), trong khi tập hợp lệ là chữ HOA → mọi số có chữ cái đều bị ERROR oan.

In [19]:
def wrong_lowercase(num):
    h = hex(int(num))[2:].replace("0", "O").replace("1", "I")
    return h if set(h) <= set("ABCDEFIO") else "ERROR"

print("sai :", wrong_lowercase("700896"))
print("đúng:", Solution().toHexspeak("700896"))

sai : ERROR
đúng: ABIEO


### Lỗi 3: Duyệt chuỗi `num` (thập phân) thay vì chuỗi hex

Kiểm tra chữ số của `"257"` (2, 5, 7) là sai hoàn toàn — phải đổi sang hex trước.

In [20]:
def wrong_check_decimal(num):
    return "ERROR" if any(c in "23456789" for c in num) else num.replace("0", "O").replace("1", "I")

print("sai :", wrong_check_decimal("257"), "| đúng:", Solution().toHexspeak("257"))
print("sai :", wrong_check_decimal("10"), "| đúng:", Solution().toHexspeak("10"))

sai : ERROR | đúng: IOI
sai : IO | đúng: A


### Lỗi 4: Quên đảo ngược khi tự đổi cơ số

`divmod` lấy chữ số **từ phải sang trái**. Quên `reversed` → chuỗi bị ngược (`"ABIEO"` thành `"OEIBA"`). Với ví dụ đề `"IOI"` đối xứng nên lỗi bị **che giấu** — luôn test thêm số không đối xứng!

In [21]:
def wrong_no_reverse(num):
    to_char = {0: "O", 1: "I", 10: "A", 11: "B", 12: "C", 13: "D", 14: "E", 15: "F"}
    n, chars = int(num), []
    while n > 0:
        n, d = divmod(n, 16)
        if d not in to_char:
            return "ERROR"
        chars.append(to_char[d])
    return "".join(chars)                   # SAI: thiếu reversed

print("đề 1 vẫn qua:", wrong_no_reverse("257"))
print("sai :", wrong_no_reverse("700896"), "| đúng:", Solution().toHexspeak("700896"))

đề 1 vẫn qua: IOI
sai : OEIBA | đúng: ABIEO


### Lỗi 5: Thay `'1'` trước rồi `'0'`… có sao không?

Ở bài này **không sao** vì `'O'` và `'I'` không bị thay tiếp. Nhưng nếu quy tắc dây chuyền (ví dụ `A→B` rồi `B→C`), thứ tự `replace` nối tiếp sẽ gây "thay 2 lần". `translate` thay **đồng thời** nên an toàn hơn.

In [22]:
s = "AB"
print(s.replace("A", "B").replace("B", "C"))      # 'CC'  <- A bị thay 2 lần
print(s.translate(str.maketrans("AB", "BC")))    # 'BC'  <- thay đồng thời

CC
BC


## 15. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Đổi sang cơ số b" → vòng `while n > 0: n, d = divmod(n, b)` + đảo ngược; hoặc hàm có sẵn `bin`, `oct`, `hex`, `format(n, "b"/"o"/"x"/"X")`.
- "Chuỗi chỉ gồm ký tự trong tập S" → `set(s) <= S` hoặc `all(c in S for c in s)`.
- "Thay nhiều ký tự" → `str.translate` + `str.maketrans`.

**Bài liên quan:**
- LeetCode 405 *Convert a Number to Hexadecimal* (có số âm, bù 2).
- LeetCode 504 *Base 7*, LeetCode 168 *Excel Sheet Column Title* (cơ số 26 lệch 1).
- LeetCode 1017 *Convert to Base -2*.

In [23]:
# Mini: hàm đổi cơ số tổng quát (2 <= base <= 16), n >= 0
def to_base(n: int, base: int) -> str:
    if n == 0:
        return "0"
    digits = "0123456789ABCDEF"
    out = []
    while n > 0:
        n, d = divmod(n, base)
        out.append(digits[d])
    return "".join(reversed(out))

print(to_base(257, 16), to_base(10, 2), to_base(100, 7))    # 101 1010 202
assert all(to_base(v, 16) == format(v, "X") for v in range(0, 5000))
assert all(to_base(v, 2) == format(v, "b") for v in range(0, 5000))
print("to_base khớp format() với 0..4999")

101 1010 202
to_base khớp format() với 0..4999


## 16. Bài tập tự luyện

> Gợi ý ở mục 17, đáp án ở mục 18. Không gọi các hàm TODO khi chạy notebook.

### Bài 1 — Tính tay
Đổi `4011` sang hex bằng tay (dùng `divmod` 16 nhiều lần), rồi cho biết Hexspeak. Gán chuỗi kết quả vào `ex1_answer`.

In [24]:
ex1_answer = None   # TODO

# assert ex1_answer == Solution().toHexspeak("4011")

### Bài 2 — Hexspeak → số thập phân
Viết `from_hexspeak(s)` nhận chuỗi Hexspeak hợp lệ và trả về chuỗi thập phân ban đầu.

In [25]:
def from_hexspeak(s: str) -> str:
    # TODO: thay O->0, I->1 rồi int(..., 16)
    pass

# assert from_hexspeak("IOI") == "257"
# assert from_hexspeak("COFFEE") == "12648430"

### Bài 3 — Đếm số Hexspeak hợp lệ
Viết `count_valid(lo, hi)` đếm số `N` trong `[lo, hi]` có Hexspeak hợp lệ (duyệt thẳng, `hi - lo` nhỏ).

In [26]:
def count_valid(lo: int, hi: int) -> int:
    # TODO
    pass

# assert count_valid(1, 16) == 8      # 1, 10..15, 16

### Bài 4 — Base-7 "speak"
Quy tắc tương tự nhưng ở **cơ số 7**: chữ số `0→O`, `1→I`, `6→G`; các chữ số khác → `"ERROR"`. Viết `to_base7speak(num: str) -> str` bằng `divmod`.

In [27]:
def to_base7speak(num: str) -> str:
    # TODO
    pass

# assert to_base7speak("7") == "IO"        # 7 = 10 (cơ số 7)
# assert to_base7speak("48") == "GG"       # 48 = 66 (cơ số 7)
# assert to_base7speak("2") == "ERROR"

### Bài 5 — Mini challenge: sinh tất cả Hexspeak có đúng `k` ký tự
Viết `all_hexspeak_of_length(k)` trả về list (tăng dần theo giá trị) các chuỗi Hexspeak hợp lệ có đúng `k` ký tự. Lưu ý ký tự đầu không thể là `O` (vì không có số 0 ở đầu).

In [28]:
def all_hexspeak_of_length(k: int) -> list[str]:
    # TODO: dùng itertools.product hoặc đệ quy
    pass

# assert all_hexspeak_of_length(1) == ["I", "A", "B", "C", "D", "E", "F"]
# assert len(all_hexspeak_of_length(2)) == 7 * 8

## 17. Gợi ý 3 mức

**Bài 1:** (1) Lặp `divmod(·, 16)`. (2) `4011 = 250·16 + 11`, `250 = 15·16 + 10`, `15 = 0·16 + 15`. (3) Hex `FAB` → kiểm tra từng chữ.

**Bài 2:** (1) Làm ngược đề. (2) `s.translate(str.maketrans("OI", "01"))`. (3) `str(int(h, 16))`.

**Bài 3:** (1) Dùng lại `Solution().toHexspeak`. (2) Duyệt `range(lo, hi + 1)`. (3) Đếm kết quả khác `"ERROR"`.

**Bài 4:** (1) Như Cách 2, đổi 16 thành 7. (2) Dict `{0: "O", 1: "I", 6: "G"}`. (3) Nhớ `reversed`.

**Bài 5:** (1) Ký tự đầu: 7 lựa chọn, các ký tự sau: 8 lựa chọn. (2) `itertools.product(chars, repeat=k-1)` sinh mọi tổ hợp. (3) Sắp xếp theo giá trị: `key=lambda s: int(s.translate(...), 16)`.

## 18. Đáp án tham khảo

In [29]:
from itertools import product

ans_ex1 = "FAB"     # 4011 = 0xFAB


def ans_from_hexspeak(s):
    return str(int(s.translate(str.maketrans("OI", "01")), 16))


def ans_count_valid(lo, hi):
    sol = Solution()
    return sum(1 for v in range(lo, hi + 1) if sol.toHexspeak(str(v)) != "ERROR")


def ans_to_base7speak(num):
    to_char = {0: "O", 1: "I", 6: "G"}
    n, out = int(num), []
    while n > 0:
        n, d = divmod(n, 7)
        if d not in to_char:
            return "ERROR"
        out.append(to_char[d])
    return "".join(reversed(out))


def ans_all_hexspeak_of_length(k):
    first = "IABCDEF"
    rest = "OIABCDEF"
    res = [f + "".join(t) for f in first for t in product(rest, repeat=k - 1)]
    table = str.maketrans("OI", "01")
    return sorted(res, key=lambda s: int(s.translate(table), 16))


assert ans_ex1 == Solution().toHexspeak("4011")
assert ans_from_hexspeak("IOI") == "257"
assert ans_from_hexspeak("COFFEE") == "12648430"
assert ans_count_valid(1, 16) == 8
assert ans_to_base7speak("7") == "IO"
assert ans_to_base7speak("48") == "GG"
assert ans_to_base7speak("2") == "ERROR"
assert ans_all_hexspeak_of_length(1) == ["I", "A", "B", "C", "D", "E", "F"]
assert len(ans_all_hexspeak_of_length(2)) == 7 * 8
# khứ hồi: from_hexspeak(toHexspeak(x)) == x với mọi Hexspeak độ dài 3
for s in ans_all_hexspeak_of_length(3):
    assert Solution().toHexspeak(ans_from_hexspeak(s)) == s
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


`lambda s: ...` là **hàm vô danh** viết trên một dòng; `sorted(..., key=f)` sắp xếp theo giá trị `f(phần_tử)`. `product(rest, repeat=k-1)` sinh mọi bộ `k-1` ký tự lấy từ `rest`.

## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def toHexspeak(self, num: str) -> str:
        to_char = {0: "O", 1: "I", 10: "A", 11: "B", 12: "C", 13: "D", 14: "E", 15: "F"}
        n, chars = int(num), []
        while n > 0:
            n, d = divmod(n, 16)
            if d not in to_char:
                return "ERROR"
            chars.append(to_char[d])
        return "".join(reversed(chars))
```

**Câu thần chú:** *"int → divmod 16 → map (0→O, 1→I, 10..15→A..F) → gặp 2..9 thì ERROR → đảo ngược."*

**Checklist:**
- [ ] Đổi `num` (chuỗi) sang `int` trước.
- [ ] Nếu dùng `hex()`: cắt `'0x'` và `.upper()`.
- [ ] Nếu tự đổi cơ số: nhớ **đảo ngược**.
- [ ] Trả đúng chuỗi `"ERROR"` (viết hoa).
- [ ] Test số không đối xứng (ví dụ `"700896"`), số có chữ số `2..9` ở giữa.